# 获取结构化结果方式

## 1.使用with_structured_output()

举例：

In [17]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model = "deepseek-flash",
    model_provider = "deepseek",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
        extra_body = {
        "thinking":{"type":"disabled"}
    }
)

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="电影导演")
    rating: float = Field(description="评分（10分制）")

structured_model = model.with_structured_output(Movie)

res = structured_model.invoke("帮我介绍一下《红高粱》")
rprint(res)
rprint(type(res))

In [ ]:
from pydantic import BaseModel, Field
from rich import print as rprint

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="电影导演")
    rating: float = Field(description="评分（10分制）")

structured_model = model.with_structured_output(Movie, include_raw=True)

res = structured_model.invoke("帮我介绍一下《红高粱》")
rprint(res)   # 输出结果中会包含AIMessage
rprint(type(res))

## 2.使用输出解析器（了解）

举例：

In [23]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint

# 1. 创建提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system","回答用户问题,必须始终输出一个包含title(电影标题)和year(上映年份)的 JSON 对象"),
    ("human","问题：{question}")
])

# 2. 模型初始化

# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

# 3. 定义结构
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")

# 4. 创建输出解析器
parser = JsonOutputParser(pydantic_object=Movie)

# 5. 创建链
chain = prompt_template | model | parser

# 6. 调用（返回字典）
# response = chain.invoke({"question": "介绍电影《盗梦空间》"})

response = parser.invoke(model.invoke(prompt_template.invoke({"question": "介绍电影《盗梦空间》"})))

rprint(type(response))
rprint(response)

<class 'dict'>

{
    'title': '盗梦空间',
    'year': 2010,
    'director': '克里斯托弗·诺兰',
    'genre': ['科幻', '悬疑', '动作'],
    'description': 
'《盗梦空间》讲述由莱昂纳多·迪卡普里奥饰演的道姆·柯布带领团队潜入他人梦境，执行植入思想的任务。影片以多层梦境、时间
流逝差异和现实边界为核心，融合动作、悬疑与哲学思考，是2010年上映的经典科幻电影。'
}